# Hybrid Quantum CVRP Solver using Aer-backed QITE
## Quantum Imaginary Time Evolution for Vehicle Routing (No Brute-Force Fallback)

This notebook runs a **QITE-style** solver for the route-ordering subproblem in CVRP using
a **Qiskit Aer statevector simulator**. The notebook has been rewritten so that:

1. **There is no brute-force TSP fallback inside the QITE solver**
2. **There is no brute-force objective-trace fallback**
3. **Route solving always attempts the QITE workflow, even when it is not practical**
4. **The provided benchmark instances are used directly for testing**

### Important note
Imaginary-time evolution is **non-unitary**, so Aer cannot implement it as a standard gate-only circuit.
In this notebook, we therefore:
- construct the Ising Hamiltonian from the TSP QUBO,
- apply exact imaginary-time propagation numerically with repeated `exp(-Δτ H)` steps,
- then load the evolved state into a **Qiskit Aer statevector simulator** for decoding and probability analysis.

This keeps the workflow **Aer-backed** while avoiding a brute-force route-order search.

### Pipeline
1. **Capacity-aware sweep partitioning** to assign customers to vehicles
2. **TSP QUBO construction** for each vehicle cluster
3. **Imaginary-time evolution** on the Ising Hamiltonian
4. **Aer statevector decoding** of the evolved state into a valid route
5. **Feasibility checks and summaries** on the provided instances


In [ ]:
import importlib
import subprocess
import sys

_REQUIRED_PACKAGES = [
    ("qiskit", "qiskit"),
    ("qiskit-aer", "qiskit_aer"),
    ("qiskit-algorithms", "qiskit_algorithms"),
    ("scipy", "scipy"),
]

for pip_name, import_name in _REQUIRED_PACKAGES:
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

import numpy as np
import math
import time
import json
import warnings
warnings.filterwarnings('ignore')

from scipy.linalg import expm

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer import AerSimulator


In [ ]:
# == Utility Functions ==

def euclidean_distance(p1, p2):
    return math.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

def build_distance_matrix(nodes):
    n = len(nodes)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            D[i,j] = euclidean_distance(nodes[i], nodes[j])
    return D


## Phase 1: Capacity-Aware Partitioning
This version does **not** use exhaustive partition enumeration.
Instead, it assigns customers to vehicles with a simple **angular sweep partition** so that
the notebook stays focused on the **QITE route-ordering step** and avoids brute-force fallback logic.


In [ ]:
import math

# --- QAOA Agglomerative Clustering Functions ---
def two_opt_cost(cluster, nodes):
    if not cluster or len(cluster) <= 1:
        c = cluster[0] if cluster else 0
        return 2 * math.sqrt((nodes[0][0]-nodes[c][0])**2 + (nodes[0][1]-nodes[c][1])**2) if cluster else 0
    def d(a, b): return math.sqrt((nodes[a][0]-nodes[b][0])**2 + (nodes[a][1]-nodes[b][1])**2)
    route = [0]
    remaining = list(cluster)
    while remaining:
        last = route[-1]
        nearest = min(remaining, key=lambda c: d(last, c))
        route.append(nearest)
        remaining.remove(nearest)
    route.append(0)
    improved = True
    while improved:
        improved = False
        for i in range(1, len(route) - 2):
            for j in range(i + 1, len(route) - 1):
                old = d(route[i-1], route[i]) + d(route[j], route[j+1])
                new = d(route[i-1], route[j]) + d(route[i], route[j+1])
                if new < old - 1e-10:
                    route[i:j+1] = reversed(route[i:j+1])
                    improved = True
    return sum(d(route[k], route[k+1]) for k in range(len(route)-1))

def agglomerative_clusters(nodes, Nv, C):
    n = len(nodes)
    D = build_distance_matrix(nodes)
    clusters = [[i] for i in range(1, n)]
    def nn_cost(cluster):
        if len(cluster) == 1: return 2 * D[0, cluster[0]]
        cost, current, remaining = 0, 0, set(cluster)
        while remaining:
            nxt = min(remaining, key=lambda c: D[current, c])
            cost += D[current, nxt]
            current = nxt
            remaining.remove(nxt)
        return cost + D[current, 0]
    while len(clusters) > 1:
        best_benefit, best_pair = -float('inf'), None
        for a in range(len(clusters)):
            for b in range(a + 1, len(clusters)):
                if len(clusters[a]) + len(clusters[b]) > C: continue
                benefit = nn_cost(clusters[a]) + nn_cost(clusters[b]) - nn_cost(clusters[a] + clusters[b])
                if benefit > best_benefit:
                    best_benefit = benefit
                    best_pair = (a, b)
        if best_pair is None: break
        if best_benefit <= 0 and len(clusters) <= Nv: break
        a, b = best_pair
        clusters[a] = clusters[a] + clusters[b]
        clusters.pop(b)
    return clusters

def refine_clusters_swap(clusters, nodes, C, func):
    clusters = [list(c) for c in clusters]
    improved = True
    iteration = 0
    while improved:
        improved = False
        iteration += 1
        for i in range(len(clusters)):
            for j in range(i+1, len(clusters)):
                old_cost = func(clusters[i], nodes) + func(clusters[j], nodes)
                best_gain, best_swap = 0, None
                for ai, a in enumerate(clusters[i]):
                    for bi, b in enumerate(clusters[j]):
                        ci_new = clusters[i][:ai] + [b] + clusters[i][ai+1:]
                        cj_new = clusters[j][:bi] + [a] + clusters[j][bi+1:]
                        if len(ci_new) > C or len(cj_new) > C: continue
                        new_cost = func(ci_new, nodes) + func(cj_new, nodes)
                        gain = old_cost - new_cost
                        if gain > best_gain:
                            best_gain, best_swap = gain, (ai, bi, ci_new, cj_new)
                for ai, a in enumerate(clusters[i]):
                    if len(clusters[j]) < C:
                        ci_new = clusters[i][:ai] + clusters[i][ai+1:]
                        cj_new = clusters[j] + [a]
                        if ci_new:
                            new_cost = func(ci_new, nodes) + func(cj_new, nodes)
                            gain = old_cost - new_cost
                            if gain > best_gain:
                                best_gain, best_swap = gain, (-1, -1, ci_new, cj_new)
                for bi, b in enumerate(clusters[j]):
                    if len(clusters[i]) < C:
                        cj_new = clusters[j][:bi] + clusters[j][bi+1:]
                        ci_new = clusters[i] + [b]
                        if cj_new:
                            new_cost = func(ci_new, nodes) + func(cj_new, nodes)
                            gain = old_cost - new_cost
                            if gain > best_gain:
                                best_gain, best_swap = gain, (-1, -1, ci_new, cj_new)
                if best_swap and best_gain > 1e-10:
                    clusters[i], clusters[j] = best_swap[2], best_swap[3]
                    improved = True
    return clusters


## Phase 2: QUBO Construction & Ising Mapping
Build a position-formulation QUBO for the TSP within each cluster, then convert to an Ising Hamiltonian.


In [ ]:
def build_tsp_qubo(cluster_indices, dist_matrix):
    """Build upper-triangular QUBO for TSP (position formulation).
    Variables x[i,s] = 1 if cluster node i is at route position s."""
    n = len(cluster_indices)
    N = n * n
    Q = np.zeros((N, N))
    idx = lambda i, s: i * n + s
    max_d = max(dist_matrix[a, b]
                for a in [0] + cluster_indices
                for b in [0] + cluster_indices if a != b)
    A = max_d * n * 1.5  # Penalty coefficient

    # Constraint: each customer visited exactly once
    for i in range(n):
        for s in range(n): Q[idx(i,s), idx(i,s)] -= A
        for s in range(n):
            for t in range(s+1, n): Q[idx(i,s), idx(i,t)] += 2 * A

    # Constraint: each position filled by exactly one customer
    for s in range(n):
        for i in range(n): Q[idx(i,s), idx(i,s)] -= A
        for i in range(n):
            for j in range(i+1, n): Q[idx(i,s), idx(j,s)] += 2 * A

    # Objective: route distance
    for i, ci in enumerate(cluster_indices):
        Q[idx(i,0), idx(i,0)]     += dist_matrix[0, ci]
        Q[idx(i,n-1), idx(i,n-1)] += dist_matrix[ci, 0]
        for j, cj in enumerate(cluster_indices):
            if i != j:
                for s in range(n - 1):
                    a, b = idx(i, s), idx(j, s+1)
                    if a <= b: Q[a, b] += dist_matrix[ci, cj]
                    else: Q[b, a] += dist_matrix[ci, cj]
    return Q

def qubo_to_ising(Q):
    """Convert QUBO to SparsePauliOp Ising Hamiltonian."""
    n = Q.shape[0]
    constant, h, J = 0.0, np.zeros(n), {}
    for k in range(n): constant += Q[k,k]/2; h[k] -= Q[k,k]/2
    for k in range(n):
        for l in range(k+1, n):
            if abs(Q[k,l]) > 1e-12:
                constant += Q[k,l]/4; h[k] -= Q[k,l]/4; h[l] -= Q[k,l]/4
                J[(k,l)] = Q[k,l]/4
    pauli_list = []
    for k in range(n):
        if abs(h[k]) > 1e-12:
            lbl = ['I']*n; lbl[n-1-k] = 'Z'
            pauli_list.append((''.join(lbl), h[k]))
    for (k,l), coef in J.items():
        if abs(coef) > 1e-12:
            lbl = ['I']*n; lbl[n-1-k] = 'Z'; lbl[n-1-l] = 'Z'
            pauli_list.append((''.join(lbl), coef))
    if not pauli_list: pauli_list = [('I'*n, 0.0)]
    return SparsePauliOp.from_list(pauli_list).simplify(), constant

def decode_bitstring(bitstring, n, cluster_indices, dist_matrix):
    """Decode measurement bitstring to ordered route."""
    bits = [int(b) for b in reversed(bitstring)]
    while len(bits) < n*n: bits.append(0)
    x = np.zeros((n, n), dtype=int)
    for i in range(n):
        for s in range(n): x[i,s] = bits[i*n+s]
    if not all(x[i,:].sum()==1 for i in range(n)): return None, float('inf')
    if not all(x[:,s].sum()==1 for s in range(n)): return None, float('inf')
    order = [0]*n
    for i in range(n): order[int(np.argmax(x[i,:]))] = cluster_indices[i]
    d = dist_matrix[0, order[0]]
    for k in range(n-1): d += dist_matrix[order[k], order[k+1]]
    d += dist_matrix[order[-1], 0]
    return order, d


## Phase 3: Aer-Backed QITE Solver
The solver below always attempts the QITE route solve and does **not** switch to brute force.

Because imaginary-time evolution is non-unitary, the evolved state is computed numerically via
repeated applications of `exp(-Δτ H)` and then loaded into **Qiskit Aer** for statevector-based
probability extraction and decoding.


In [ ]:
USE_AER_GPU = False  # Set to False to force CPU

try:
    if USE_AER_GPU:
        _test_sim = AerSimulator(method='statevector', device='GPU')
        _test_sim.status()
        AER_BACKEND = AerSimulator(method='statevector', device='GPU')
        print("Aer GPU backend successfully initialized!")
    else:
        AER_BACKEND = AerSimulator(method="statevector")
        print("Aer CPU backend initialized (GPU flag is False).")
except Exception as e:
    AER_BACKEND = AerSimulator(method="statevector")
    print(f"Aer GPU unavailable, falling back to CPU. ({e})")


def evolve_state_imaginary_time(hamiltonian, time_param=10.0, num_timesteps=50):
    n_qubits = hamiltonian.num_qubits
    dim = 2 ** n_qubits
    H_mat = np.asarray(hamiltonian.to_matrix(), dtype=complex)

    psi = np.ones(dim, dtype=complex) / np.sqrt(dim)
    steps = max(1, int(num_timesteps))
    dt = float(time_param) / steps
    propagator = expm(-dt * H_mat)

    energy_trace = []
    for _ in range(steps):
        psi = propagator @ psi
        norm = np.linalg.norm(psi)
        if norm == 0:
            raise RuntimeError("Imaginary-time propagation collapsed to the zero vector.")
        psi = psi / norm
        energy = float(np.real(np.vdot(psi, H_mat @ psi)))
        energy_trace.append(energy)

    return psi, energy_trace


def aer_probabilities_from_statevector(statevector):
    num_qubits = int(np.log2(len(statevector)))
    qc = QuantumCircuit(num_qubits)
    qc.initialize(statevector, range(num_qubits))
    qc.save_statevector()
    tqc = transpile(qc, AER_BACKEND)
    result = AER_BACKEND.run(tqc).result()
    evolved_sv = Statevector(result.get_statevector(tqc))
    return evolved_sv.probabilities_dict()


def solve_tsp_qite(cluster_indices, dist_matrix, time_param=10.0, num_timesteps=50):
    if len(cluster_indices) == 0:
        return [], 0.0, 0, 0.0, []
    if len(cluster_indices) == 1:
        node = cluster_indices[0]
        val = 2 * dist_matrix[0, node]
        return [node], val, 1, 0.0, [val]*num_timesteps

    t0 = time.time()
    n = len(cluster_indices)
    Q = build_tsp_qubo(cluster_indices, dist_matrix)
    H, offset = qubo_to_ising(Q)

    evolved_state, energy_trace = evolve_state_imaginary_time(
        H,
        time_param=time_param,
        num_timesteps=num_timesteps,
    )
    
    full_energy_trace = [e + offset for e in energy_trace]

    prob_dict = aer_probabilities_from_statevector(evolved_state)

    best_route = None
    best_dist = float("inf")

    for bitstring, prob in sorted(prob_dict.items(), key=lambda item: item[1], reverse=True):
        if prob <= 0.0:
            continue
        route, dist = decode_bitstring(bitstring, n, cluster_indices, dist_matrix)
        if route is not None and dist < best_dist:
            best_route = route
            best_dist = dist

    elapsed = time.time() - t0
    return best_route, best_dist, H.num_qubits, elapsed, full_energy_trace


Aer GPU unavailable, falling back to CPU. ('AerSimulator' object has no attribute 'status')


## Load Official Instances


In [ ]:
import json

# --- Load Instances from final_instances.json ---
with open('../final_instances.json', 'r') as f:
    official_instances = json.load(f)

for config in official_instances:
    inst_id = config['instance_id']
    n_cust = len(config['customers']) - 1
    print(f"  {inst_id}: {n_cust} customers, {config['Nv']} vehicles, capacity {config['C']}")


FileNotFoundError: [Errno 2] No such file or directory: 'final_instances.json'

## Run Aer-Backed QITE on the Provided Instances
For each provided instance, we:
- build a sweep-based capacity partition,
- run the Aer-backed QITE solver on each vehicle cluster,
- report the total route length and feasibility.

There is **no brute-force TSP fallback** in this run.


In [ ]:
# Create globals to store across both cells
all_results = []
all_convergence = {}

# --- Execute Instances 1 to 4 ---
for config in official_instances[:4]:
    inst_id = config['instance_id']
    Nv = config['Nv']
    C = config['C']
    
    if isinstance(config['customers'][0], dict):
        nodes = [(float(c['x']), float(c['y'])) for c in config['customers']]
    else:
        nodes = [(float(c[0]), float(c[1])) for c in config['customers']]
        
    n_cust = len(nodes) - 1
    D = build_distance_matrix(nodes)

    print(f"\n{'='*65}")
    print(f"  {inst_id}: {n_cust} customers, {Nv} vehicles, capacity {C}")
    print(f"{'='*65}")

    clusters = agglomerative_clusters(nodes, Nv, C)
    opt_partition = refine_clusters_swap(clusters, nodes, C, two_opt_cost)

    print(f"  Agglomerative Clusters: {opt_partition}")
    print(f"\n  Running Aer-backed QITE...")
    
    routes = []
    total_dist = 0
    total_time = 0
    max_qubits = 0

    opt_cost = sum(two_opt_cost(grp, nodes) for grp in opt_partition)
    
    # Track Convergence Plot Data
    log = []
    cumulative_evals = 0
    settled_energy = 0.0

    for grp in opt_partition:
        route, dist, nq, et, trace = solve_tsp_qite(grp, D, time_param=10.0, num_timesteps=50)
        
        curr_min = float("inf")
        # Build the cumulative objective plot exactly like QAOA!
        for e in trace:
            cumulative_evals += 1
            curr_min = min(curr_min, e)
            log.append((cumulative_evals, settled_energy + curr_min))
            
        settled_energy += curr_min
        
        full_route = [0] + route + [0]
        routes.append(full_route)
        total_dist += dist
        total_time += et
        max_qubits = max(max_qubits, nq)
        print(f"    Vehicle: {' -> '.join(map(str, full_route))}  dist={dist:.4f}  qubits={nq}  time={et:.2f}s")
        
    all_convergence[inst_id] = log

    gap = total_dist - opt_cost
    print(f"\n  QITE Total Distance: {total_dist:.4f}")
    print(f"  Classical Estimate:  {opt_cost:.4f}")
    print(f"  QITE - Reference Gap:{gap:.4f}")

    all_results.append({
        'id': inst_id, 'nodes': nodes, 'routes': routes,
        'total': total_dist, 'reference': opt_cost, 'gap': gap,
        'time': total_time, 'qubits': max_qubits, 'Nv': Nv, 'C': C,
        'n_cust': n_cust, 'partition': opt_partition, 'reference_routes': opt_partition
    })



  1: 3 customers, 2 vehicles, capacity 5
  Agglomerative Clusters: [[1, 2, 3]]

  Running Aer-backed QITE...
    Vehicle: 0 -> 3 -> 2 -> 1 -> 0  dist=21.7445  qubits=9  time=0.34s

  QITE Total Distance: 21.7445
  Classical Estimate:  21.7445
  QITE - Reference Gap:0.0000

  2: 3 customers, 2 vehicles, capacity 2
  Agglomerative Clusters: [[1, 2], [3]]

  Running Aer-backed QITE...
    Vehicle: 0 -> 1 -> 2 -> 0  dist=18.9706  qubits=4  time=0.04s
    Vehicle: 0 -> 3 -> 0  dist=7.2111  qubits=1  time=0.00s

  QITE Total Distance: 26.1817
  Classical Estimate:  26.1817
  QITE - Reference Gap:0.0000

  3: 6 customers, 3 vehicles, capacity 2
  Agglomerative Clusters: [[1, 2], [3, 6], [4, 5]]

  Running Aer-backed QITE...
    Vehicle: 0 -> 1 -> 2 -> 0  dist=18.9706  qubits=4  time=0.05s
    Vehicle: 0 -> 6 -> 3 -> 0  dist=13.2111  qubits=4  time=0.05s
    Vehicle: 0 -> 5 -> 4 -> 0  dist=17.3171  qubits=4  time=0.05s

  QITE Total Distance: 49.4988
  Classical Estimate:  49.4988
  QITE - Re

## Results Summary

In [10]:
print(f"\n{'='*86}")
print(f"{'QITE RESULTS SUMMARY':^86}")
print(f"{'='*86}")
print(
    f"{'Instance':>12} | {'Cust':>4} | {'Veh':>3} | {'Cap':>3} | "
    f"{'QITE Dist':>10} | {'Sweep Ref':>10} | {'Gap':>10} | {'Qubits':>6} | {'Time':>8}"
)
print(f"{'-'*86}")

for r in all_results:
    print(
        f"{r['id']:>12} | {r['n_cust']:>4} | {r['Nv']:>3} | {r['C']:>3} | "
        f"{r['total']:>10.4f} | {r['reference']:>10.4f} | {r['gap']:>10.4f} | "
        f"{r['qubits']:>6} | {r['time']:>7.2f}s"
    )

print(f"{'='*86}")



                                 QITE RESULTS SUMMARY                                 
    Instance | Cust | Veh | Cap |  QITE Dist |  Sweep Ref |        Gap | Qubits |     Time
--------------------------------------------------------------------------------------
           1 |    3 |   2 |   5 |    21.7445 |    21.7445 |     0.0000 |      9 |    0.34s
           2 |    3 |   2 |   2 |    26.1817 |    26.1817 |     0.0000 |      4 |    0.04s
           3 |    6 |   3 |   2 |    49.4988 |    49.4988 |     0.0000 |      4 |    0.14s
           4 |   12 |   4 |   3 |    58.1783 |    58.1783 |     0.0000 |      9 |    0.29s


In [ ]:
# Use the Variables pane or `dir()` in a scratch cell if you need to inspect the kernel.

In [ ]:
import time
import csv

# Set True to regenerate scaling_results.csv.
RUN_SCALING_BENCHMARK = False

try:
    import pandas as pd
except ImportError:
    pd = None

results = []
num_trials = 2


def clusters_for_capacity(nodes, Nv, C):
    """Partition customers into routes using the same sweep strategy as the main benchmark."""
    customers = list(range(1, len(nodes)))
    D = build_distance_matrix(nodes)
    partition, _, _ = sweep_partition(customers, Nv, C, D, nodes)
    return partition


if RUN_SCALING_BENCHMARK:
    for config in official_instances:
        inst_id = config["instance_id"]
        Nv = config["Nv"]

        try:
            nodes = [(c["x"], c["y"]) for c in config["customers"]]
        except (KeyError, TypeError):
            print(f"Skipping {inst_id} (bad format)")
            continue

        base_C = config["C"]
        print(f"\n=== {inst_id} ===")

        for C in range(base_C, base_C + 3):
            print(f"  C = {C}")

            for trial in range(num_trials):
                try:
                    t0 = time.time()
                    clusters = clusters_for_capacity(nodes, Nv, C)
                    D = build_distance_matrix(nodes)

                    total_cost = 0.0
                    for cluster in clusters:
                        if not cluster:
                            continue
                        _, cost, *_ = solve_tsp_qite(cluster, D)
                        total_cost += cost

                    runtime = time.time() - t0
                    results.append({
                        "instance": inst_id,
                        "trial": trial,
                        "capacity": C,
                        "distance": total_cost,
                        "runtime": runtime,
                    })
                except Exception as e:
                    print(f"    trial {trial} failed: {e}")
                    continue

    if pd is not None:
        df = pd.DataFrame(results)
        df.to_csv("scaling_results.csv", index=False)
        print("\nDONE. Saved to scaling_results.csv")
        print(df.head())
    else:
        with open("scaling_results.csv", "w", newline="") as f:
            w = csv.DictWriter(
                f,
                fieldnames=["instance", "trial", "capacity", "distance", "runtime"],
            )
            w.writeheader()
            w.writerows(results)
        print("\nDONE. Saved to scaling_results.csv (csv; install pandas for DataFrame)")
else:
    print("Skipping scaling benchmark (set RUN_SCALING_BENCHMARK = True to regenerate CSV).")


In [ ]:
# Run one selected JSON instance through Aer-backed QITE
from pathlib import Path
import json

json_instance_file = "new_instance_2.json"
json_instance_id = "instance_5"
json_print_selected_instance = False
json_qite_time_param = 10.0
json_qite_num_timesteps = 50


def load_qite_json_instance(file_path, instance_id):
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(
            f"JSON instance file not found: {path}. Update json_instance_file and rerun this cell."
        )

    with path.open("r", encoding="utf-8") as fh:
        payload = json.load(fh)

    if not isinstance(payload, list):
        raise ValueError("Expected a top-level JSON list of instances.")

    selected = next(
        (item for item in payload if item.get("instance_id") == instance_id),
        None,
    )
    if selected is None:
        available = [item.get("instance_id") for item in payload if isinstance(item, dict)]
        raise ValueError(
            f"Instance ID '{instance_id}' not found in {path.name}. Available IDs: {available}"
        )

    customers = selected.get("customers")
    if not isinstance(customers, list) or not customers:
        raise ValueError("Selected instance must contain a non-empty 'customers' list.")

    customer_map = {}
    for customer in customers:
        if not isinstance(customer, dict):
            raise ValueError("Each customer entry must be a JSON object.")
        if "customer_id" not in customer or "x" not in customer or "y" not in customer:
            raise ValueError("Each customer must define customer_id, x, and y.")
        customer_id = int(customer["customer_id"])
        customer_map[customer_id] = {
            "customer_id": customer_id,
            "x": float(customer["x"]),
            "y": float(customer["y"]),
            "demand": int(customer.get("demand", 1)),
        }

    ordered_ids = sorted(customer_map)
    if not ordered_ids or ordered_ids[0] != 0 or ordered_ids != list(range(ordered_ids[-1] + 1)):
        raise ValueError("Customer IDs must be contiguous and start at 0.")

    normalized_customers = [customer_map[idx] for idx in ordered_ids]
    nodes = [(customer_map[idx]["x"], customer_map[idx]["y"]) for idx in ordered_ids]

    normalized_instance = {
        "instance_id": str(selected["instance_id"]),
        "Nv": int(selected["Nv"]),
        "C": int(selected["C"]),
        "customers": normalized_customers,
    }
    return normalized_instance, nodes, int(selected["Nv"]), int(selected["C"])


def print_simple_table(title, headers, rows):
    rows = [[str(value) for value in row] for row in rows]
    widths = [len(str(header)) for header in headers]
    for row in rows:
        for idx, value in enumerate(row):
            widths[idx] = max(widths[idx], len(value))

    total_width = sum(widths) + 3 * (len(headers) - 1)
    print(f"\n{'=' * total_width}")
    print(f"{title:^{total_width}}")
    print(f"{'=' * total_width}")
    print(" | ".join(f"{header:>{widths[idx]}}" for idx, header in enumerate(headers)))
    print("-" * total_width)
    for row in rows:
        print(" | ".join(f"{value:>{widths[idx]}}" for idx, value in enumerate(row)))
    print("=" * total_width)


def run_selected_qite_json_instance(
    file_path,
    instance_id,
    print_selected_instance=False,
    time_param=10.0,
    num_timesteps=50,
):
    instance_json, nodes, Nv, C = load_qite_json_instance(file_path, instance_id)
    D = build_distance_matrix(nodes)
    customers = list(range(1, len(nodes)))
    n_cust = len(customers)
    reference_label = "Sweep NN Ref"

    print(f"\n{'=' * 75}")
    print(f"{'SELECTED AER-QITE JSON RUN':^75}")
    print(f"{'=' * 75}")

    input_rows = [
        ["File", Path(file_path).name],
        ["Instance", instance_id],
        ["Customers", n_cust],
        ["Vehicles", Nv],
        ["Capacity", C],
        ["Time Param", f"{time_param:.2f}"],
        ["Timesteps", num_timesteps],
    ]
    print_simple_table("QITE INPUT", ["Field", "Value"], input_rows)

    if print_selected_instance:
        print("\nSelected instance JSON:")
        print(json.dumps(instance_json, indent=2))

    partition, ref_routes, reference_cost = sweep_partition(customers, Nv, C, D, nodes)

    routes = []
    route_details = []
    total_dist = 0.0
    total_time = 0.0
    max_qubits = 0

    for vehicle_idx, group in enumerate(partition, start=1):
        route, dist, nq, elapsed = solve_tsp_qite(
            group,
            D,
            time_param=time_param,
            num_timesteps=num_timesteps,
        )
        full_route = [0] + route + [0] if route else [0, 0]
        routes.append(full_route)
        total_dist += dist
        total_time += elapsed
        max_qubits = max(max_qubits, nq)
        route_details.append(
            {
                "vehicle": vehicle_idx,
                "route": " -> ".join(map(str, full_route)),
                "customers": len(group),
                "distance": dist,
                "qubits": nq,
                "time": elapsed,
            }
        )

    gap = total_dist - reference_cost

    summary_rows = [
        [
            instance_id,
            n_cust,
            Nv,
            C,
            f"{total_dist:.4f}",
            f"{reference_cost:.4f}",
            f"{gap:.4f}",
            max_qubits,
            f"{total_time:.2f}s",
        ]
    ]
    print_simple_table(
        "QITE RESULT SUMMARY",
        ["Instance", "Cust", "Veh", "Cap", "QITE Dist", reference_label, "Gap", "Qubits", "Time"],
        summary_rows,
    )

    route_rows = [
        [
            detail["vehicle"],
            detail["customers"],
            detail["route"],
            f"{detail['distance']:.4f}",
            detail["qubits"],
            f"{detail['time']:.2f}s",
        ]
        for detail in route_details
    ]
    print_simple_table(
        "QITE ROUTES",
        ["Vehicle", "Cust", "Route", "Dist", "Qubits", "Time"],
        route_rows,
    )

    return {
        "instance_id": instance_id,
        "file_path": str(Path(file_path)),
        "instance_json": instance_json,
        "partition": partition,
        "reference_routes": ref_routes,
        "routes": routes,
        "route_details": route_details,
        "total_distance": total_dist,
        "reference_distance": reference_cost,
        "reference_label": reference_label,
        "gap": gap,
        "max_qubits": max_qubits,
        "total_time": total_time,
        "Nv": Nv,
        "C": C,
        "n_customers": n_cust,
        "time_param": time_param,
        "num_timesteps": num_timesteps,
    }


json_qite_result = run_selected_qite_json_instance(
    file_path=json_instance_file,
    instance_id=json_instance_id,
    print_selected_instance=json_print_selected_instance,
    time_param=json_qite_time_param,
    num_timesteps=json_qite_num_timesteps,
)


In [ ]:
# Plot cumulative energy vs shots from a convergence CSV
from pathlib import Path
import os
import csv

cache_dir = Path.cwd() / ".cache"
mpl_cache_dir = Path.cwd() / ".matplotlib-cache"
os.environ.setdefault("XDG_CACHE_HOME", str(cache_dir))
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache_dir))
cache_dir.mkdir(parents=True, exist_ok=True)
mpl_cache_dir.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt

convergence_source_csv = "convergence_4.csv"
convergence_instance_label = "instance_4"
convergence_plot_title = "QAOA Convergence — Instance 4"
shots_per_evaluation = 1
output_dir = Path.cwd() / "generated"
output_plot_csv = output_dir / f"{convergence_instance_label}_shots_cumulative_energy.csv"
output_plot_png = output_dir / f"{convergence_instance_label}_shots_cumulative_energy.png"


def load_convergence_rows(csv_path, shots_per_eval=1):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Convergence CSV not found: {csv_path}. Update convergence_source_csv and rerun this cell."
        )

    with csv_path.open("r", encoding="utf-8", newline="") as fh:
        reader = csv.DictReader(fh)
        fieldnames = reader.fieldnames or []
        if "objective_value" not in fieldnames:
            raise ValueError(
                f"CSV must include 'objective_value'. Found columns: {fieldnames}"
            )

        rows = []
        running_best = float("inf")
        for idx, row in enumerate(reader, start=1):
            if "shots" in row and row["shots"] not in (None, ""):
                shots = float(row["shots"])
            elif "circuit_evaluation" in row and row["circuit_evaluation"] not in (None, ""):
                shots = float(row["circuit_evaluation"]) * float(shots_per_eval)
            else:
                shots = float(idx) * float(shots_per_eval)

            objective_value = float(row["objective_value"])
            running_best = min(running_best, objective_value)

            rows.append(
                {
                    "circuit_evaluation": int(float(row.get("circuit_evaluation", idx))),
                    "shots": shots,
                    "objective_value": objective_value,
                    "cumulative_energy": running_best,
                }
            )

    if not rows:
        raise ValueError(f"No data rows found in {csv_path}.")
    return rows


def save_convergence_csv(rows, csv_path):
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", encoding="utf-8", newline="") as fh:
        writer = csv.DictWriter(
            fh,
            fieldnames=[
                "circuit_evaluation",
                "shots",
                "objective_value",
                "cumulative_energy",
            ],
        )
        writer.writeheader()
        writer.writerows(rows)


def plot_convergence(rows, plot_path, plot_title):
    plot_path = Path(plot_path)
    plot_path.parent.mkdir(parents=True, exist_ok=True)

    shots = [row["shots"] for row in rows]
    cumulative = [row["cumulative_energy"] for row in rows]

    plt.figure(figsize=(8, 4))
    plt.step(shots, cumulative, where="post", linewidth=1.5, color="#1f77b4")
    plt.title(plot_title)
    plt.xlabel("Shots")
    plt.ylabel("Cumulative Energy")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()


def print_convergence_summary(rows, plot_csv, plot_png):
    start_row = rows[0]
    end_row = rows[-1]
    best_row = min(rows, key=lambda row: row["cumulative_energy"])
    summary = [
        ("Input Rows", len(rows)),
        ("First Eval", start_row["circuit_evaluation"]),
        ("Last Eval", end_row["circuit_evaluation"]),
        ("First Shots", f"{start_row['shots']:.0f}"),
        ("Last Shots", f"{end_row['shots']:.0f}"),
        ("Start Energy", f"{start_row['objective_value']:.4f}"),
        ("Best Cumulative Energy", f"{best_row['cumulative_energy']:.4f}"),
        ("Saved CSV", str(plot_csv)),
        ("Saved PNG", str(plot_png)),
    ]

    if "print_simple_table" in globals():
        print_simple_table("CONVERGENCE OUTPUTS", ["Field", "Value"], summary)
    else:
        print("\nConvergence Outputs")
        print("-" * 80)
        for field, value in summary:
            print(f"{field:>24}: {value}")


convergence_rows = load_convergence_rows(
    convergence_source_csv,
    shots_per_eval=shots_per_evaluation,
)
save_convergence_csv(convergence_rows, output_plot_csv)
plot_convergence(convergence_rows, output_plot_png, convergence_plot_title)
print_convergence_summary(convergence_rows, output_plot_csv, output_plot_png)


In [ ]:
# Plot QITE cumulative objective over imaginary-time steps
from pathlib import Path
import os
import csv

cache_dir = Path.cwd() / ".cache"
mpl_cache_dir = Path.cwd() / ".matplotlib-cache"
os.environ.setdefault("XDG_CACHE_HOME", str(cache_dir))
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache_dir))
cache_dir.mkdir(parents=True, exist_ok=True)
mpl_cache_dir.mkdir(parents=True, exist_ok=True)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

qite_plot_instance_file = "new_instance_2.json"
qite_plot_instance_id = "instance_1"
qite_plot_time_param = 10.0
qite_plot_num_timesteps = 20
qite_plot_output_dir = Path.cwd() / "generated"
qite_plot_prefix = f"{qite_plot_instance_id}_qite_cumulative_objective"


def load_qite_plot_instance(file_path, instance_id):
    path = Path(file_path)
    with path.open("r", encoding="utf-8") as fh:
        payload = json.load(fh)

    selected = next(
        (item for item in payload if item.get("instance_id") == instance_id),
        None,
    )
    if selected is None:
        available = [item.get("instance_id") for item in payload if isinstance(item, dict)]
        raise ValueError(
            f"Instance ID '{instance_id}' not found in {path.name}. Available IDs: {available}"
        )

    customer_map = {int(c["customer_id"]): c for c in selected["customers"]}
    ordered_ids = sorted(customer_map)
    if ordered_ids != list(range(ordered_ids[-1] + 1)):
        raise ValueError("Customer IDs must be contiguous starting at 0.")

    nodes = [(float(customer_map[idx]["x"]), float(customer_map[idx]["y"])) for idx in ordered_ids]
    return selected, nodes, int(selected["Nv"]), int(selected["C"])


def qite_cluster_objective_trace(cluster_indices, dist_matrix, time_param=10.0, num_timesteps=20):
    if not cluster_indices:
        return [0.0] * max(1, num_timesteps), 0

    if len(cluster_indices) == 1:
        value = 2 * dist_matrix[0, cluster_indices[0]]
        return [value] * max(1, num_timesteps), 1

    Q = build_tsp_qubo(cluster_indices, dist_matrix)
    H, offset = qubo_to_ising(Q)
    _, energy_trace = evolve_state_imaginary_time(
        H,
        time_param=time_param,
        num_timesteps=num_timesteps,
    )
    objective_trace = [energy + offset for energy in energy_trace]
    return objective_trace, H.num_qubits


def build_qite_cumulative_objective(instance_file, instance_id, time_param=10.0, num_timesteps=20):
    _, nodes, Nv, C = load_qite_plot_instance(instance_file, instance_id)
    D = build_distance_matrix(nodes)
    customers = list(range(1, len(nodes)))
    partition, _, reference_cost = sweep_partition(customers, Nv, C, D, nodes)

    cluster_traces = []
    cluster_qubits = []
    for vehicle_idx, cluster in enumerate(partition, start=1):
        trace, n_qubits = qite_cluster_objective_trace(
            cluster,
            D,
            time_param=time_param,
            num_timesteps=num_timesteps,
        )
        running_min = []
        curr_min = float("inf")
        for value in trace:
            curr_min = min(curr_min, value)
            running_min.append(curr_min)
        cluster_traces.append(
            {
                "vehicle": vehicle_idx,
                "cluster": list(cluster),
                "trace": trace,
                "running_min": running_min,
            }
        )
        cluster_qubits.append(n_qubits)

    max_len = max(len(item["running_min"]) for item in cluster_traces) if cluster_traces else 0
    padded = [
        item["running_min"] + [item["running_min"][-1]] * (max_len - len(item["running_min"]))
        for item in cluster_traces
    ] if cluster_traces else []
    total_obj = [
        sum(padded[c][step] for c in range(len(padded)))
        for step in range(max_len)
    ] if padded else []

    rows = []
    for step_idx, value in enumerate(total_obj, start=1):
        rows.append(
            {
                "imaginary_time_step": step_idx,
                "cumulative_objective": value,
            }
        )

    summary = {
        "instance_id": instance_id,
        "vehicles": Nv,
        "capacity": C,
        "customers": len(customers),
        "reference_cost": reference_cost,
        "cluster_traces": cluster_traces,
        "max_cluster_qubits": max(cluster_qubits) if cluster_qubits else 0,
        "total_objective_trace": total_obj,
        "rows": rows,
    }
    return summary


def save_qite_cumulative_objective_outputs(summary, output_dir, prefix):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_path = output_dir / f"{prefix}.csv"
    png_path = output_dir / f"{prefix}.png"

    with csv_path.open("w", encoding="utf-8", newline="") as fh:
        writer = csv.DictWriter(
            fh,
            fieldnames=["imaginary_time_step", "cumulative_objective"],
        )
        writer.writeheader()
        writer.writerows(summary["rows"])

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(
        [row["imaginary_time_step"] for row in summary["rows"]],
        [row["cumulative_objective"] for row in summary["rows"]],
        linewidth=2,
    )
    ax.set_title(f"QITE Cumulative Objective — {summary['instance_id']}")
    ax.set_xlabel("Imaginary-Time Step")
    ax.set_ylabel("Objective Value (total route cost)")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return csv_path, png_path


qite_cumulative_summary = build_qite_cumulative_objective(
    instance_file=qite_plot_instance_file,
    instance_id=qite_plot_instance_id,
    time_param=qite_plot_time_param,
    num_timesteps=qite_plot_num_timesteps,
)
qite_cumulative_csv, qite_cumulative_png = save_qite_cumulative_objective_outputs(
    qite_cumulative_summary,
    qite_plot_output_dir,
    qite_plot_prefix,
)

if "print_simple_table" in globals():
    print_simple_table(
        "QITE CUMULATIVE OBJECTIVE",
        ["Field", "Value"],
        [
            ["Instance", qite_cumulative_summary["instance_id"]],
            ["Customers", qite_cumulative_summary["customers"]],
            ["Vehicles", qite_cumulative_summary["vehicles"]],
            ["Capacity", qite_cumulative_summary["capacity"]],
            ["Sweep NN Reference", f"{qite_cumulative_summary['reference_cost']:.4f}"],
            ["Final Cumulative Objective", f"{qite_cumulative_summary['total_objective_trace'][-1]:.4f}" if qite_cumulative_summary["total_objective_trace"] else "n/a"],
            ["Max Cluster Qubits", qite_cumulative_summary["max_cluster_qubits"]],
            ["Saved CSV", str(qite_cumulative_csv)],
            ["Saved PNG", str(qite_cumulative_png)],
        ],
    )
else:
    print("QITE cumulative objective outputs:")
    print(qite_cumulative_csv)
    print(qite_cumulative_png)
